In [5]:
import os
import re
import json
import glob
import joblib
import pandas as pd
import numpy as np
from datetime import timedelta

try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip install catboost -q
    from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# ==========================================
# 1. THE MELTER (FIXES YOUR 51-COLUMN FILES)
# =========================
def process_soccer_mon_file(path):
    tag = re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", os.path.basename(path).split('.')[0].lower())).strip("_")

    try:
        df = pd.read_csv(path) if path.endswith('.csv') else pd.read_json(path)
    except:
        return None, None

    # Detect if file is WIDE (Date + 50 Player columns) or LONG (player_name column)
    cols = [c.lower() for c in df.columns]

    # CASE A: LONG FORMAT (Injury, Game-Performance, Illness)
    if 'player_name' in cols or 'player_id' in cols:
        p_col = 'player_name' if 'player_name' in cols else 'player_id'
        d_col = 'timestamp' if 'timestamp' in cols else [c for c in cols if 'date' in c or 'time' in c][0]

        df = df.rename(columns={p_col: 'player_id', d_col: 'date'})
        df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce').dt.normalize()
        df = df.dropna(subset=['date', 'player_id'])
        return df, "long"

    # CASE B: WIDE FORMAT (Wellness, ACWR, CTL, etc.)
    else:
        # First column is usually the date
        d_col = df.columns[0]
        df[d_col] = pd.to_datetime(df[d_col], dayfirst=True, errors='coerce').dt.normalize()
        df = df.dropna(subset=[d_col])

        # Melt: Convert column headers into a 'player_id' column
        melted = df.melt(id_vars=[d_col], var_name='player_id', value_name=f'{tag}_value')
        melted = melted.rename(columns={d_col: 'date'})
        return melted, "wide"

# ==========================================
# 2. AGGREGATION & MERGING
# =========================
all_files = glob.glob("*.csv") + glob.glob("*.json")
feature_dfs = []
injury_df = None

print("🔄 Melting and Merging 19 files...")

for path in all_files:
    fname = os.path.basename(path)
    df_clean, fmt = process_soccer_mon_file(path)

    if df_clean is None: continue

    if 'injury' in fname.lower():
        injury_df = df_clean[['player_id', 'date']].drop_duplicates()
        print(f"✅ Loaded Injury Log: {len(injury_df)} records")
    else:
        # Standardize ID as string to ensure join works
        df_clean['player_id'] = df_clean['player_id'].astype(str)
        feature_dfs.append(df_clean)

# Combine all wellness/load metrics
master = feature_dfs[0]
for df in feature_dfs[1:]:
    master = pd.merge(master, df, on=['player_id', 'date'], how='outer')

# Ensure IDs are clean (UUIDs can have spaces or quotes)
master['player_id'] = master['player_id'].str.strip()
if injury_df is not None:
    injury_df['player_id'] = injury_df['player_id'].str.strip()

# ==========================================
# 3. TARGETING (The Overlap Check)
# =========================
master['is_risk'] = 0
if injury_df is not None:
    for _, row in injury_df.iterrows():
        # Tag 7 days before an injury as High Risk
        mask = (master['player_id'] == row['player_id']) & \
               (master['date'] >= row['date'] - timedelta(days=7)) & \
               (master['date'] < row['date'])
        master.loc[mask, 'is_risk'] = 1

print(f"📈 Risk distribution after melting: {master['is_risk'].sum()} high-risk days found.")

if master['is_risk'].sum() < 2:
    print("⚠️ WARNING: Very few injuries overlapped. Check if dates in wellness files match injury years.")

# ==========================================
# 4. TRAINING
# =========================
X = master.drop(columns=['player_id', 'date', 'is_risk'])
y = master['is_risk']

# Clean any weird nested strings like '["nausea"]'
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].astype(str).str.replace(r'[\[\]"]', '', regex=True)

# Split (Skip stratify if too few samples)
strat = y if y.sum() > 5 else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=strat, random_state=42)

model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    eval_metric='AUC',
    verbose=100
)

# Detect categories
cat_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
model.fit(X_train, y_train, eval_set=(X_test, y_test), cat_features=cat_features)

# ==========================================
# 5. EXPORT
# =========================
joblib.dump({"model": model, "features": X.columns.tolist()}, "model_risk.pkl")
with open("risk_schema.json", "w") as f:
    json.dump({"features": X.columns.tolist()}, f)

print("🎉 DONE! model_risk.pkl is ready.")

🔄 Melting and Merging 19 files...
✅ Loaded Injury Log: 156 records
📈 Risk distribution after melting: 538 high-risk days found.
0:	test: 0.7913305	best: 0.7913305 (0)	total: 83.3ms	remaining: 1m 6s
100:	test: 0.8676175	best: 0.8676175 (100)	total: 3.31s	remaining: 22.9s
200:	test: 0.8714868	best: 0.8766480 (172)	total: 6.3s	remaining: 18.8s
300:	test: 0.8920094	best: 0.8952205 (292)	total: 9.48s	remaining: 15.7s
400:	test: 0.8851938	best: 0.8962142 (306)	total: 15s	remaining: 14.9s
500:	test: 0.8802782	best: 0.8962142 (306)	total: 18.1s	remaining: 10.8s
600:	test: 0.8771005	best: 0.8962142 (306)	total: 21.3s	remaining: 7.07s
700:	test: 0.8787459	best: 0.8962142 (306)	total: 24.6s	remaining: 3.48s
799:	test: 0.8776224	best: 0.8962142 (306)	total: 30s	remaining: 0us

bestTest = 0.8962141546
bestIteration = 306

Shrink model to first 307 iterations.
🎉 DONE! model_risk.pkl is ready.


In [6]:
import joblib
import pandas as pd
import numpy as np

# Load the artifacts you just generated in Colab
bundle = joblib.load("model_risk.pkl")
model = bundle["model"]
required_features = bundle["features"]

def predict_injury_risk(new_wellness_data, player_id):
    """
    new_wellness_data: A dictionary or JSON of today's metrics
    Example: {'fatigue': 4, 'sleep_quality': 3, 'acwr': 1.2, ...}
    """
    # 1. Convert to DataFrame
    df = pd.DataFrame([new_wellness_data])

    # 2. Reconstruct the feature names expected by the model
    # The model expects 'fatigue_value', 'sleep_quality_value', etc.
    processed_data = {}
    for feat in required_features:
        # Check if the feature is a base value or a 7d_avg
        base_name = feat.replace("_value", "").replace("_7d_avg", "")

        if "_7d_avg" in feat:
            # For a real system, you'd fetch the last 7 days from your DB
            # For now, we'll use the current value as a fallback
            processed_data[feat] = new_wellness_data.get(base_name, 0)
        else:
            processed_data[feat] = new_wellness_data.get(base_name, 0)

    input_df = pd.DataFrame([processed_data])

    # 3. Predict Probability
    probability = model.predict_proba(input_df)[0][1]

    # 4. Interpret Result
    status = "Low Risk"
    if probability > 0.75:
        status = "High Risk"
    elif probability > 0.45:
        status = "Medium Risk"

    return {
        "player_id": player_id,
        "risk_score": round(float(probability), 2),
        "status": status
    }

# --- TEST ---
# example_player_data = {'fatigue': 5, 'sleep_quality': 2, 'acwr': 1.8}
# print(predict_injury_risk(example_player_data, "Player_A1"))

In [7]:
from google.colab import files
files.download('model_risk.pkl')
files.download('risk_schema.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>